# Machine Learning Pt.1 Activity
## BTC1895H Python
## Lauren MacIntyre

# Insurance Dataset Workshop

## Goal
Understand the dataset, identify the ML problem, select a target, and justify the approach before modeling.

In [ ]:
import pandas as pd
import numpy as np

In [8]:
# Loading the Insurance data set
df = pd.read_csv("insurance_mystery.csv")
df.head()

,age,bmi,smoker,exercise_level,claims_history,blood_pressure,family_history,premium_quote,branch_code,advisor_id,category_flag
0,76,23.8,0,4,4,128.4,0,307.4,W1,10201,high
1,54,26.7,0,4,2,119.7,0,186.2,N1,10228,medium
2,64,16.0,0,3,2,128.4,0,93.8,S1,10040,low
3,51,23.9,0,5,1,100.3,0,99.5,N1,10145,low
4,51,27.0,0,3,1,159.9,0,96.1,W1,10164,low


In [9]:
# Inital data exploration and overview
df.head()              # shows first few rows of the dataset
df.info()              # displays column types and non-null counts
df.describe()          # summary statistics for numerical (and some categorical) data
df.isnull().sum()      # counts missing values in each column

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             520 non-null    int64  
 1   bmi             520 non-null    float64
 2   smoker          520 non-null    int64  
 3   exercise_level  520 non-null    int64  
 4   claims_history  520 non-null    int64  
 5   blood_pressure  520 non-null    float64
 6   family_history  520 non-null    int64  
 7   premium_quote   520 non-null    float64
 8   branch_code     520 non-null    object 
 9   advisor_id      520 non-null    int64  
 10  category_flag   520 non-null    object 
dtypes: float64(3), int64(6), object(2)
memory usage: 44.8+ KB


age               0
bmi               0
smoker            0
exercise_level    0
claims_history    0
blood_pressure    0
family_history    0
premium_quote     0
branch_code       0
advisor_id        0
category_flag     0
dtype: int64

## Part 1 - Problem Understanding
Dataset name: insurance_mystery.csv

### 1. What do you think this dataset is about?

This dataset appears to be about insurance customers or applicants and the factors used to estimate their risk level or pricing. It includes health, lifestyle, and administrative information that may be used to assign a premium quote or risk category.

### 2. What type of ML problem is this?

**Answer:** Not sure yet

**Why?**  
At this stage, the dataset could support multiple types of machine learning problems. The presence of `premium_quote` suggests a regression problem, while `category_flag` suggests a classification problem. I need to further examine the dataset and choose a target variable before definitively deciding the problem type.

### 3. What do you think is the TARGET variable?

**Column name:** `category_flag`

**Why did you choose this?**  
I selected `category_flag` because it represents a clear labeled outcome with categories such as low, medium, and high. This makes it suitable for a classification problem.

### 4. Which columns are likely USEFUL features?

Likely useful features include:

- `age`
- `bmi`
- `smoker`
- `exercise_level`
- `claims_history`
- `blood_pressure`

These variables are likely useful because they relate to a person’s health, lifestyle, and past insurance risk, which would reasonably influence their risk category or pricing.

### 5. Which columns look SUSPICIOUS or USELESS?

`premium_quote` looks suspicious because the `category_flag` may be derived directly from it. Including it as a feature could introduce data leakage and make the model appear artificially accurate.

`advisor_id` also appears to be useless because it likely functions as an identifier rather than a meaningful predictor of insurance risk.

`branch_code` may also be less useful, as it represents administrative information rather than individual risk factors.

## Part 2 - Build the Model

### 6. What model will you try first?

**Answer:** Random Forest

**Why?**  
I will try a Random Forest first because it works well for classification problems, can handle non-linear relationships, and is a good baseline model without requiring too many assumptions about the data. It is also useful here because the dataset includes a mix of numeric and categorical variables.

In [10]:
# Build the Model
# the questions were answered in this file, but this is the githib link anyways
# https://github.com/laur1111/ml-workshop-1-BTC1895H/blob/main/machine_learning_insurance.ipynb


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# define features and target
X = df.drop(columns=["category_flag", "premium_quote", "advisor_id"])  # drop target + suspicious/useless columns
y = df["category_flag"]                                                # target variable

# identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# preprocessing for numeric and categorical data
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))   # fills missing numeric values if needed
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),   # fills missing categorical values if needed
    ("onehot", OneHotEncoder(handle_unknown="ignore"))      # converts text categories into numeric columns
])

# combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# create train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# build modeling pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# fit model
model.fit(X_train, y_train)

# make predictions
y_pred = model.predict(X_test)

# evaluate model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.5961538461538461

Confusion Matrix:
 [[22  1  6]
 [ 3 25  9]
 [ 9 14 15]]

Classification Report:
               precision    recall  f1-score   support

        high       0.65      0.76      0.70        29
         low       0.62      0.68      0.65        37
      medium       0.50      0.39      0.44        38

    accuracy                           0.60       104
   macro avg       0.59      0.61      0.60       104
weighted avg       0.59      0.60      0.59       104



## Part 3 - Evaluate the Model

### 8. What is your result?

Accuracy / Score: 0.60

### 9. Do you trust this result?

Not sure 

**Why?**  
The model achieves about 60% accuracy, which suggests it is learning some patterns but is not highly accurate. The performance also varies across classes, with the model performing better on "high" and "low" categories than "medium". This suggests the model may not generalize equally well across all groups, so the result should be interpreted cautiously.

### 10. What could go wrong?

Data leakage, wrong model choice, bad features  

**Explain:**  

There is a risk of data leakage if any features indirectly reflect how the `category_flag` was assigned. For example, although `premium_quote` was removed, other variables could still be closely related to pricing decisions, which may inflate model performance.

The model choice may also not be optimal. A Random Forest is a good baseline, but it may not fully capture the relationships in the data, especially if the classes are not easily separable. Trying other models (such as logistic regression or tuning hyperparameters) could improve performance.

Some features may also be weak or not directly relevant to predicting risk. For example, administrative variables like `branch_code` may not reflect true individual risk, and including less informative features can reduce model accuracy.

Additionally, the model performs noticeably worse on the "medium" category compared to "low" and "high", suggesting that the classes may overlap or be harder to distinguish. This could limit overall performance regardless of the model used.

## Part 4 - Improve Your Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# build new model using a Decision Tree instead of Random Forest
model2 = Pipeline(steps=[
    ("preprocessor", preprocessor),                          # apply same preprocessing as before
    ("classifier", DecisionTreeClassifier(random_state=42))  # simpler classification model
])

# train the model on the training data
model2.fit(X_train, y_train)

# make predictions on the test set
y_pred2 = model2.predict(X_test)

# evaluate model performance
print("Accuracy:", accuracy_score(y_test, y_pred2))      # overall accuracy
print("\nClassification Report:\n", classification_report(y_test, y_pred2))  # precision, recall, f1-score

Accuracy: 0.4807692307692308

Classification Report:
               precision    recall  f1-score   support

        high       0.55      0.55      0.55        29
         low       0.54      0.54      0.54        37
      medium       0.37      0.37      0.37        38

    accuracy                           0.48       104
   macro avg       0.49      0.49      0.49       104
weighted avg       0.48      0.48      0.48       104



### 11. What did you try to improve the model?

Changed the model  

**What exactly did you do?**  
I replaced the Random Forest model with a Decision Tree classifier to see whether a simpler model would perform better on the dataset.

### 12. New result:

Score: 0.48

### 13. Did it improve?

No, it did not

**Why?**  
The model performance decreased from about 0.60 to 0.48 when using a Decision Tree. This suggests that the Random Forest model was better at capturing patterns in the data. A Random Forest combines multiple trees and reduces overfitting, whereas a single Decision Tree is more limited and can perform worse on complex datasets.

## PART 5: Data Scientist Thinking

### 14. What is ONE key insight from this dataset?

One key insight is that health and lifestyle factors such as BMI, smoking status, and claims history appear to influence the risk category, but the model struggles to clearly separate the “medium” group from “low” and “high”.

### 15. If you were to explain to a business person, what would you say?

This dataset can be used to help classify customers into risk categories based on their health and behaviour. The model shows that factors like smoking, BMI, and past claims are important, but it is not highly accurate yet, especially for customers in the middle risk group. This suggests that while the model could support decision-making, it would need further improvement before being used in practice.

### 16. What was the hardest part?

The hardest part was deciding on the target variable and recognizing the potential for data leakage. It required careful reasoning to avoid using variables like `premium_quote` that could make the model appear artificially accurate.